Detecteur de Déchets 

In [5]:
import tensorflow as tf
import os

IMG_SIZE = 640 # Défini par le prétraitement Roboflow
NUM_CLASSES = 11 # Défini dans data.yaml

def parse_single_yolo_label(label_path):
    """
    Lit un fichier texte YOLO contenant un seul objet. type [class_id, x_center, y_center, width, height]
    Retourne la boîte englobante (y_min, x_min, y_max, x_max) et l'ID de la classe.
    """
    # Lecture du fichier
    text = tf.io.read_file(label_path)
    lines = tf.strings.split(text, sep='\n')
    
    # Si le fichier est vide (faille dans le dataset), on retourne des valeurs par défaut
    def process_line():
        parts = tf.strings.split(lines[0])
        cls_id = tf.strings.to_number(parts[0], out_type=tf.int32)
        x_center = tf.strings.to_number(parts[1], out_type=tf.float32)
        y_center = tf.strings.to_number(parts[2], out_type=tf.float32)
        width = tf.strings.to_number(parts[3], out_type=tf.float32)
        height = tf.strings.to_number(parts[4], out_type=tf.float32)

        x_min = x_center - (width / 2.0)
        y_min = y_center - (height / 2.0)
        x_max = x_center + (width / 2.0)
        y_max = y_center + (height / 2.0)
        
        return tf.stack([y_min, x_min, y_max, x_max]), cls_id

    def empty_default():
        return tf.zeros([4], dtype=tf.float32), tf.constant(0, dtype=tf.int32)

    return tf.cond(tf.size(lines) > 0, process_line, empty_default)

def load_image_and_labels(image_path):
    # 1. Image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3) 
    img = tf.cast(img, tf.float32) / 255.0 
    img = tf.reshape(img, [IMG_SIZE, IMG_SIZE, 3])
    
    # 2. Label
    # Remplacement conditionnel pour gérer les chemins de train, valid et test
    label_path = tf.strings.regex_replace(image_path, "images", "labels")
    label_path = tf.strings.regex_replace(label_path, r"\.jpg$", ".txt")
    
    bbox, cls_id = parse_single_yolo_label(label_path)
    
    # 3. Format de sortie ciblé
    # On renvoie un dictionnaire pour les étiquettes car notre futur réseau aura deux sorties
    return img, {"bbox_output": bbox, "class_output": cls_id}

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def build_dataset(split_name):
    file_pattern = f"{split_name}/images/*.jpg" 
    
    dataset = tf.data.Dataset.list_files(file_pattern, shuffle=(split_name == "train"))
    dataset = dataset.map(load_image_and_labels, num_parallel_calls=AUTOTUNE)
    
    if split_name == "train":
        # Cache en mémoire après le premier epoch pour accélérer les suivants
        dataset = dataset.cache()
        dataset = dataset.shuffle(1000)
        
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset

train_ds = build_dataset("data/train")
val_ds   = build_dataset("data/valid")
test_ds  = build_dataset("data/test")

# Vérification
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("BBox shape:", labels["bbox_output"].shape)
    print("Class shape:", labels["class_output"].shape)

W0000 00:00:1791961941.109386  117812 cpu_allocator_impl.cc:82] Allocation of 157286400 exceeds 10% of free system memory.
W0000 00:00:1791961941.292159  117810 cpu_allocator_impl.cc:82] Allocation of 157286400 exceeds 10% of free system memory.
W0000 00:00:1791961941.304029  117813 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 157287040 bytes after encountering the first element of size 157287040 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
W0000 00:00:1791961941.307515  115917 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 157287040 bytes after encountering the first element of size 157287040 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


Images shape: (32, 640, 640, 3)
BBox shape: (32, 4)
Class shape: (32,)
